# Deployment 5 — calibration data (load / noise states)

Standalone look at the switched-calibration data: `RFAMB` (ambient load,
13.2k integrations / 134 files) and `RFNON` (noise source, 7.6k / 101 files),
selected per-integration with `eigsep_data.MetadataIndex`. Bulk of it is the
Jul 17 cal cycle (per file: ~84 ant → ~110 noise → ~42 load rows); scattered
visits Jul 13/14/16/18. Load thermistor (`tempctrl_load` `T_now`, the
`tempctrl_load_T_now` column) works (~26.6 °C); `tempctrl_lna` reports errors
all deployment.

Row times are the index's per-integration `time_best`, not the one file-close
time the `rfswitch_index.npz` prototype gave every row of a file, so visit
boundaries and the stability-vs-time plot are sharp to the integration.

Keys: `0` = box-gnd, `4` = box-air (phase C). Rows before Jul 15 are the
same physical switch but different antenna wiring upstream, and never
recorded key `0` at all — they load as NaN and the `nanmean`s below skip
them. Add `filter_phase="C"` (or a `files=` pattern) to the selection to
keep to one wiring phase.

In [ ]:
%matplotlib widget
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from eigsep_data import MetadataIndex

# repo root = first parent dir containing the filtered data
REPO = Path.cwd().resolve()
while not (REPO / "data" / "deployment5_filtered").is_dir():
    if REPO.parent == REPO:
        raise FileNotFoundError("data/deployment5_filtered not found above cwd")
    REPO = REPO.parent
DATA_DIR = REPO / "data" / "deployment5_filtered"

# A cold scan of the deployment the first time, cached after:
# ~65 s for the 5120 files / 1.23M rows, then ~5 s from the sidecar.
idx = MetadataIndex(DATA_DIR)

In [ ]:
KEYS = ["0", "4"]
PATTERNS = None          # e.g. ["corr_20260717*"] for just the big cal day
GAP_S = 600              # new visit after this many seconds without cal data


def gated(state):
    """Every integration of `state`, as the dict the cells below read."""
    # Rows with no time_best (an inconsistent clock and no filename
    # stamp to fall back on) are dropped on purpose: everything below is
    # ordered in time, and Selection.visits() gives such rows the id -1
    # rather than inventing a place for them. summary() reports what
    # each filter, this one included, removed.
    sel = idx.select(
        rfswitch=state, files=PATTERNS, where=lambda df: df.time_best.notna()
    )
    print(sel.summary())
    # Phases A/B never recorded key "0", so those rows come back NaN
    # instead of raising -- the same stand-in the prototype filled in,
    # and the nanmeans below skip them. Add filter_phase="C" to the
    # select above and drop missing= to keep to one wiring phase and
    # have an absent key be an error again.
    d = sel.load(keys=KEYS, missing="nan")
    return {
        "spec": {k: v.astype(np.float32) for k, v in d.data.items()},
        "t": d.times,  # per-integration time_best
        "int_t": d.meta.integration_time.to_numpy(),
        "t_load": d.meta.tempctrl_load_T_now.to_numpy(),
        "freqs": d.freq,
        # time_avg=1 keeps load() in selection order, so these visit
        # ids line up row for row with "spec" and "t".
        "visit": sel.visits(gap_s=GAP_S),
    }


amb = gated("RFAMB")
non = gated("RFNON")
freqs = amb["freqs"]
for name, c in [("RFAMB", amb), ("RFNON", non)]:
    t0 = datetime.fromtimestamp(c["t"][0], tz=timezone.utc)
    t1 = datetime.fromtimestamp(c["t"][-1], tz=timezone.utc)
    print(
        f"{name}: {len(c['t'])} rows, {c['visit'].max() + 1} visits, "
        f"{t0:%b %d %H:%M} - {t1:%b %d %H:%M} UTC"
    )

## Waterfalls (all cal rows, gated)

In [ ]:
def cal_wfall(c, key, vmin=3.5, vmax=6, title=""):
    d = c["spec"][key]
    with np.errstate(divide="ignore", invalid="ignore"):
        img = np.log10(np.abs(d))
    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    im = ax.imshow(img, aspect="auto", cmap="plasma", interpolation="none",
                   vmin=vmin, vmax=vmax,
                   extent=[freqs.min(), freqs.max(), len(d), 0])
    fig.colorbar(im, ax=ax, label="log10(power)")
    # visit boundaries + date labels
    edges = np.flatnonzero(np.diff(c["visit"])) + 1
    for row in edges:
        ax.axhline(row, color="w", ls=":", lw=0.6)
    for row in np.concatenate([[0], edges]).astype(int):
        lab = datetime.fromtimestamp(c["t"][row], tz=timezone.utc).strftime("%b %d %H:%M")
        ax.text(0.01, row, " " + lab, color="w", fontsize=7, va="top",
                transform=ax.get_yaxis_transform(),
                bbox={"facecolor": "k", "alpha": 0.4, "pad": 1, "edgecolor": "none"})
    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel("cal row")
    ax.set_title(title)

cal_wfall(amb, "0", title="RFAMB (load), key 0")
cal_wfall(non, "0", title="RFNON (noise), key 0")

## Mean spectra + per-visit spread

In [ ]:
def visit_means(c, key):
    d = np.where(c["spec"][key] > 0, c["spec"][key], np.nan)  # zero rows -> NaN
    vids = np.unique(c["visit"])
    vm = np.array([np.nanmean(d[c["visit"] == v], axis=0) for v in vids])
    vt = np.array([c["t"][c["visit"] == v].mean() for v in vids])
    return vm, vt

fig, axes = plt.subplots(1, len(KEYS), figsize=(10, 4.5), layout="constrained",
                         sharey=True)
for ax, key in zip(np.atleast_1d(axes), KEYS):
    for c, lab, col in [(amb, "load", "C0"), (non, "noise", "C1")]:
        vm, _ = visit_means(c, key)
        ax.plot(freqs, vm.T, color=col, alpha=0.15, lw=0.5)
        ax.plot(freqs, np.nanmedian(vm, axis=0), color=col, lw=1.5, label=lab)
    ax.set_yscale("log")
    ax.set_xlabel("Frequency [MHz]")
    ax.set_title(f"key {key}")
    ax.legend()
np.atleast_1d(axes)[0].set_ylabel("Power [counts]")
plt.show()

## Y-factor: noise / load

Per-visit mean noise spectrum over the *nearest-in-time* load visit mean —
the switched Y-factor. Flat + stable = healthy noise diode & receiver chain.

In [ ]:
KEY = "0"
vm_a, vt_a = visit_means(amb, KEY)
vm_n, vt_n = visit_means(non, KEY)
pair = np.abs(vt_n[:, None] - vt_a[None, :]).argmin(axis=1)  # nearest load visit
Y = vm_n / vm_a[pair]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), layout="constrained")
ax1.plot(freqs, Y.T, alpha=0.2, lw=0.5, color="C2")
ax1.plot(freqs, np.nanmedian(Y, axis=0), color="k", lw=1.5, label="median")
ax1.set_xlabel("Frequency [MHz]")
ax1.set_ylabel("noise / load")
ax1.set_title(f"Y-factor per visit, key {KEY}")
ax1.legend()

band = (freqs > 60) & (freqs < 200)
yb = np.nanmean(Y[:, band], axis=1)
tt = (vt_n - vt_n[0]) / 3600
ax2.plot(tt, yb, ".-")
ax2.set_xlabel(f"Hours since {datetime.fromtimestamp(vt_n[0], tz=timezone.utc):%b %d %H:%M} UTC")
ax2.set_ylabel("band-mean Y (60-200 MHz)")
plt.show()

## Stability vs time (with load temperature)

In [ ]:
KEY = "0"
band = (freqs > 60) & (freqs < 200)
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
axT = ax.twinx()
for c, lab, col in [(amb, "load", "C0"), (non, "noise", "C1")]:
    d = np.where(c["spec"][KEY] > 0, c["spec"][KEY], np.nan)
    p = np.nanmean(d[:, band], axis=1)
    tt = (c["t"] - amb["t"][0]) / 3600
    ax.plot(tt, p, ".", ms=2, color=col, label=lab)
    axT.plot(tt, c["t_load"], ".", ms=1, color="gray", alpha=0.5)
ax.set_yscale("log")
ax.set_xlabel(f"Hours since {datetime.fromtimestamp(amb['t'][0], tz=timezone.utc):%b %d %H:%M} UTC")
ax.set_ylabel(f"band power, key {KEY} (60-200 MHz)")
axT.set_ylabel("load T [C]", color="gray")
ax.legend()
plt.show()

## Noise-likeness within visits

Residuals about each visit's mean, over the radiometer expectation
(`mean / sqrt(2 df dt)`): Gaussian with σ ≈ 1 if the cal states are
radiometer-limited on visit timescales.

In [ ]:
def hist_gauss(x, bins=np.arange(-8, 8, 0.1), log=True, title=""):
    x = x[np.isfinite(x)]
    med = np.median(x)
    sig = 1.4826 * np.median(np.abs(x - med))
    print(f"{title}: mean={x.mean():.3f} std={x.std():.3f} | "
          f"median={med:.3f} MAD-sigma={sig:.3f}")
    gx = np.linspace(bins[0], bins[-1], 1000)
    plt.figure()
    plt.hist(x, bins=bins, density=True, histtype="step", lw=1.5, label="data")
    plt.plot(gx, np.exp(-gx**2 / 2) / np.sqrt(2 * np.pi), "k--", label="N(0, 1)")
    plt.plot(gx, np.exp(-((gx - med) / sig) ** 2 / 2) / (sig * np.sqrt(2 * np.pi)),
             "r-", label=f"N({med:.2f}, {sig:.2f}$^2$)")
    if log:
        plt.yscale("log")
        plt.ylim(1e-6, 1)
    plt.xlabel("(data - visit mean) / radiometer noise")
    plt.ylabel("density")
    plt.title(title)
    plt.legend()
    plt.show()
    return med, sig

DF_HZ = 0.244140625e6
KEY = "0"
for c, lab in [(amb, "RFAMB"), (non, "RFNON")]:
    d = np.where(c["spec"][KEY] > 0, c["spec"][KEY], np.nan)
    res = np.full_like(d, np.nan)
    for v in np.unique(c["visit"]):
        m = c["visit"] == v
        vm = np.nanmean(d[m], axis=0)
        nm = vm / np.sqrt(2 * DF_HZ * c["int_t"][m][:, None])
        res[m] = (d[m] - vm) / nm
    band = (freqs > 60) & (freqs < 200)
    hist_gauss(res[:, band].ravel(), title=f"{lab} key {KEY}")

## Scratch

In [ ]:
plt.close("all")  # run when widget figures pile up -- each holds its arrays